In [3]:
# 创建 chat model 并绑定 agent
import os

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

load_dotenv()

aliyun_base_url = os.getenv("ALIYUN_BASE_URL")
aliyun_api_key = os.getenv("ALIYUN_API_KEY")

model = init_chat_model(
    model="qwen3.7-flash-2026-07-15",
    model_provider="openai",
    base_url=aliyun_base_url,
    api_key=aliyun_api_key,
)


可以使用好几种方法来创建结构化输出：
1. 使用 系统提示词
2. 使用 create_agent 函数的 response_format 参数
3. 使用 init_chat_model 的 profile 参数指定

In [4]:

system_prompt = """
# 身份
- 你是一个 json 处理专家，可以返回纯 json 格式

# 指令
- 只返回 json 格式的数据

# 示例
user: "张三今年28岁"
assistant: {"name": "张三", "age": 28}
"""

agent = create_agent(
    model=model,
    system_prompt=system_prompt,
)

result = agent.invoke(
    {"messages": [HumanMessage("小虎今年已经 88 岁了，但是别人还是叫他小虎，但仔细想想他一辈子都叫小虎")]}
)

# 打印结果结果
print("result::: \n", result["messages"][-1].content)

result::: 
 {
  "name": "小虎",
  "age": 88
}


In [10]:
from pydantic import BaseModel, Field

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"].name)
print(result["structured_response"].email)
print(result["structured_response"].phone)


John Doe
john@example.com
(555) 123-4567


In [ ]:
from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"].name)
print(result["structured_response"].email)
print(result["structured_response"].phone)

John Doe
john@example.com
(555) 123-4567


In [ ]:
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ConstructedOutputType(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person



agent = create_agent(
    model=model,
    response_format=ConstructedOutputType,
    tools=[],
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"]["name"])
print(result["structured_response"]["email"])
print(result["structured_response"]["phone"])

John Doe
john@example.com
(555) 123-4567


In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy


constructed_output_schema = {
    "title": "ContactInfo",
    "type": "object",
    "description": "Contact information for a person.",
    "properties": {
        "name": {"type": "string", "description": "The name of the person"},
        "email": {"type": "string", "description": "The email address of the person"},
        "phone": {"type": "string", "description": "The phone number of the person"},
    },
    "required": ["name", "email", "phone"],
}


agent = create_agent(
    model=model,
    response_format=ProviderStrategy(constructed_output_schema),
    tools=[],
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567",
            }
        ]
    }
)

print(result["structured_response"]["name"])
print(result["structured_response"]["email"])
print(result["structured_response"]["phone"])

John Doe
john@example.com
(555) 123-4567


In [17]:
from dataclasses import dataclass
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


@dataclass
class ProductReview:
    """Analysis of a product review."""
    rating: int | None  # The rating of the product (1-5)
    sentiment: Literal["positive", "negative"]  # The sentiment of the review
    key_points: list[str]  # The key points of the review

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ProductReview), # 用 ToolStrategy 来包裹 结构化输出的类
    tools=[],
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})

print(result["structured_response"].rating)
print(result["structured_response"].sentiment)
print(result["structured_response"].key_points)

5
positive
['Great product', 'Fast shipping', 'Expensive']
